### Install Libraries

In [1]:
# %pip install ddgs trafilatura
%pip install openai-agents

  Using cached websockets-16.1.1-cp314-cp314-win_amd64.whl.metadata (7.0 kB)
  Using cached cryptography-50.0.1-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached cffi-2.1.1-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 25.5 MB/s  0:00:00
Using cached websockets-16.1.1-cp314-cp314-win_amd64.whl (179 kB)
Using cached cryptography-50.0.1-cp311-abi3-win_amd64.whl (3.8 MB)
Using cached cffi-2.1.1-cp314-cp314-win_amd64.whl (187 kB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ----------------------------- ---------- 5.2/7.0 MB 24.2 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 22.8 MB/s  0:00:00
Using cached 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.99.0 requires openai<3.0.0,>=2.20.0, but you have openai 3.21.0 which is incompatible.


### Setup

In [2]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [6]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got Results")
    return json.dumps(results, indent=2)

In [5]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 Got text: {len(text)} chars")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

### Step 2: System Prompt

In [7]:
#context / what it is
#tools available
#explains process roughly to give it room to think
#Let us know when it is finished, preferably with a keyword or mark
#LLMs are undeterministic so they may still not follow the prompt exactly. Some models are rated better at following directions. https://labs.scale.com/leaderboard/instruction_following.
# in general a bigger model or a reasoning model will be able to follow instructions better.

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 3: Define the Agent

In [8]:
research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

### Let's Run it!

In [9]:
result = await Runner.run(
    research_agent,
    input = "Research the following topic and produce a comprehensive research brief: how AI is used in Healthcare",
    max_turns=30
)

  ✅ Got Results
  
 ✅ Got text: 9861 chars
  ❌ Failed to fetch or extract text from https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/
  
 ✅ Got text: 98721 chars
  ✅ Got Results
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/
  ✅ Got Results
  ❌ Failed to fetch or extract text from https://www.researchgate.net/publication/389402322_Artificial_Intelligence_in_Healthcare_2024_Year_in_Review
  ❌ Failed to fetch or extract text from https://www.medrxiv.org/content/10.1101/2025.02.26.25322978v2
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC12455834/
  ✅ Got Results
  
 ✅ Got text: 29518 chars
  
 ✅ Got text: 14938 chars


In [10]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

Agent: Research Agent
---


Here is a comprehensive research brief on "How AI is Used in Healthcare" based on multiple trusted sources including Wikipedia, Cleveland Clinic, and a 2025 Menlo Ventures report among others:

---

### Research Brief: How AI is Used in Healthcare

#### Key Facts and Statistics
- Healthcare AI spending reached $1.4 billion in 2025, nearly tripling from 2024.
- 22% of healthcare organizations have deployed AI solutions, a 7-fold increase over 2024.
- Providers lead AI adoption at 27%, followed by outpatient (18%) and payers (14%).
- Healthcare is deploying AI at 2.2 times the rate of the broader U.S. economy.
- Ambient scribing (AI-generated clinical documentation) is a breakout category alone generating $600 million in 2025.
- 85% of healthcare AI spending flows to startups rather than established IT incumbents.
- AI could automate and save billions in administrative overhead, improving clinician workflow and reducing burnout.

#### Main Themes and Applications of AI in Healthcare

**1. Diagnostic Assistance and Imaging**
- AI improves early and accurate diagnosis using radiology, dermatology, pathology, and ophthalmology imaging.
- AI models can detect subtle patterns in X-rays, MRIs, CT scans, ultrasounds, and mammograms to flag abnormalities like cancer or cardiovascular diseases.
- AI in radiology helps triage urgent cases faster (e.g., stroke, pulmonary embolism) and reduce false positives.
- AI-enhanced diagnostic tools, like an AI-powered stethoscope, detect heart conditions faster than traditional methods.

**2. Clinical Decision Support Systems (CDSS)**
- AI aids physicians by providing evidence-based, real-time guidance linking patient data to medical literature and clinical guidelines.
- Tools like OpenEvidence help synthesize vast research to inform medication choices, risk assessments, and treatment plans.

**3. Personalized Treatment and Drug Discovery**
- AI analyzes genetic and clinical data to tailor therapies, improving treatment outcomes.
- AI models accelerate drug discovery by simulating molecular interactions and designing novel drugs and gene therapies.
- Notably, AI-designed antibiotics and non-psychedelic psychedelics are examples of innovations speeding new treatment development.

**4. Automated Clinical Documentation and Administrative Automation**
- Ambient AI scribes transcribe doctor-patient interactions, auto-fill electronic health records (EHR), and reduce clinician time spent on paperwork.
- AI automates insurance claims processing, billing, prior authorization, coding, and revenue cycle management, potentially recovering lost revenue and reducing delays.
- AI virtual assistants and chatbots improve patient engagement, triage symptoms, schedule appointments, and provide 24/7 patient communication support.

**5. Genomics and Rare Disease Diagnosis**
- AI models, such as PopEVE, analyze genetic variants to diagnose rare conditions more accurately and quickly.
- AI avoids bias against underrepresented populations by avoiding penalizing ancestral genetic variants unique to minority groups.

**6. Research and Regenerative Medicine**
- AI accelerates medical research by autonomously reviewing literature, hypothesizing, and analyzing data.
- AI is instrumental in synthetic biology (e.g., designing new viruses, assisting CRISPR gene editing).
- Lab-grown tissues and organs (bone marrow, lungs, bone grafts) increasingly rely on AI modeling to mimic human physiology for drug testing and transplantation innovations.

**7. Ethical, Regulatory, and Equity Challenges**
- AI implementation raises concerns about data privacy, algorithmic bias, job automation, and trust in AI-driven care.
- The U.S. and EU are progressing toward frameworks regulating healthcare AI, focusing on safety, transparency, and patient rights.
- Addressing AI bias and equitable data representation is critical to avoid exacerbating healthcare disparities.

---

### Notable Data Points
- AI diagnostic accuracy for conditions like tooth decay, skin cancer, and breast cancer is comparable or superior to expert clinicians.
- AI-powered ambient scribes aim to reduce documentation time by over 50%.
- Startups dominate innovation but incumbents (Epic, Oracle) are integrating AI to defend market position.
- Prior authorization automation could save billions for providers and patients by reducing delays.
- AI enhances early detection of heart disease, Alzheimer's, and infectious diseases.
- AI models can estimate biological age from facial images, aiding personalized clinical decisions.

---

### Source URLs
1. Wikipedia: Artificial intelligence in healthcare  
   https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare

2. Cleveland Clinic: How AI Is Used in Healthcare  
   https://health.clevelandclinic.org/ai-in-healthcare

3. Menlo Ventures, 2025 The State of AI in Healthcare  
   https://menlovc.com/perspective/2025-the-state-of-ai-in-healthcare/

---

This brief synthesizes insights from current literature and industry reports, reflecting the dynamic expansion and multifaceted impact of AI in modern healthcare delivery and research.